# 0.0. IMPORTS

In [59]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import inflection
import datetime

from IPython.display import Image

## 0.1. Helper Functions

## 0.2. Loading Data

In [48]:
df_sales_raw = pd.read_csv('data/train.csv', low_memory=False)
df_store_raw = pd.read_csv('data/store.csv', low_memory=False)

# 1.0. Passo 01 - DESCRICAO DOS DADOS

In [50]:
# join à esquerda entre os dataframes df_sales_raw e df_store_raw usando a chave primária Store
df_raw = pd.merge(df_sales_raw, df_store_raw, how='left', on='Store')

In [52]:
# cópia do merge
df1 = df_raw.copy()

## 1.1. Rename Columns

In [ ]:
# seleciona todas as colunas do dataframe df1 e armazena em cols_old
cols_old = df1.columns

# cria uma função lambda para transformar os nomes das colunas no formato snakecase
snakecase = lambda x: inflection.underscore(x)

# aplica essa função lambda em cima dos nomes das colunas antigas para realizar a transformação
cols_new = list(map(snakecase, cols_old))

# realiza a renomeação das colunas
df1.columns = cols_new

## 1.2. Data Dimensions

In [ ]:
# exibe o número de linhas e colunas do dataframe
print(f'Number of Rows: {df1.shape[0]}')
print(f'Number of Cols: {df1.shape[1]}')

## 1.3. Data Types

In [ ]:
# altera o tipo de dado da coluna date para o tipo data
df1['date'] = pd.to_datetime(df1['date'])

# exibe os tipos de dados de cada coluna do dataframe df1
df1.dtypes

## 1.4. Check NA

In [ ]:
# lista a quantidade de NA por coluna do dataframe df1
df1.isna().sum()

## 1.5. Fillout Na

In [ ]:
df1['competition_distance'].max()

In [ ]:
# competition_distance 
df1['competition_distance'] = df1['competition_distance'].apply(lambda x: 200000.0 if math.isnan(x) else x)

# para tratar os valores ausentes em competition_distance seguiu-se a convenção de que se alguma observação de competition_distance é NA
# isso significa que a loja em questão não possui concorrência próxima, por isso, decidiu-se aplicar um valor maior que o máximo de quando se tinha
# valores ausentes

# competition_open_since_month 
df1['competition_open_since_month'] = df1.apply(lambda x: x['date'].month if math.isnan( x['competition_open_since_month'] ) else x['competition_open_since_month'], axis=1)

# para tratar os valores ausentes em competition_open_since_month, entrou-se no consenso de que se não houvesse um mês de início de concorrência com
# uma loja próxima, o mês a ser imputado seria extraído da coluna date

# competition_open_since_year    
df1['competition_open_since_year'] = df1.apply(lambda x: x['date'].year if math.isnan( x['competition_open_since_year'] ) else x['competition_open_since_year'], axis=1)

# para tratar os valores ausentes em competition_open_since_year, entrou-se no consenso de que se não houvesse um ano de início de concorrência com
# uma loja próxima, o ano a ser imputado seria extraído da coluna date

# promo2_since_week  
df1['promo2_since_week'] = df1.apply(lambda x: x['date'].week if math.isnan( x['promo2_since_week'] ) else x['promo2_since_week'], axis=1)

# para tratar os valores ausentes em promo2_since_week, entrou-se no consenso de que se não houvesse uma semana de início da campanha de promoções 2,
# seria imputada a semana correspondente extraída da coluna date

# promo2_since_year 
df1['promo2_since_year'] = df1.apply(lambda x: x['date'].year if math.isnan( x['promo2_since_year'] ) else x['promo2_since_year'], axis=1)

# para tratar os valores ausentes em promo2_since_year, entrou-se no consenso de que se não houvesse um ano de início da campanha de promoções 2,
# seria imputado o ano correspondente extraído da coluna date

# promo_interval

# dicionário auxiliar responsável por mapear número do mês em nome do mês
month_map = {
    1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Apr',
    5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug',
    9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

# preencher todas as observações NA da coluna promo_interval com 0 
df1['promo_interval'] = df1['promo_interval'].fillna(0)

# extrair a informação numérica de mês da coluna date, mapear esse número para nome do mês e criar uma coluna month_map
df1['month_map'] = df1['date'].dt.month.map(month_map)

# verificar se o mês month_map está inserido em promo_interval, verificar se houve promo_interval de fato e criar uma coluna is_promo onde:
# 0 = não faz sentido que a loja esteve em promoção naquele mês
# 1 = faz sentido que a loja esteve em promoção naquele mês
df1['is_promo'] = df1[['promo_interval', 'month_map']].apply(
    lambda x: 0 if x['promo_interval'] == 0
    else 1 if x['month_map'] in x['promo_interval'].split(',')
    else 0,
    axis=1
)

In [ ]:
# lista a quantidade de NA por coluna do dataframe df1 após o tratamento de NA
df1.isna().sum()

## 1.6. Change Types

In [ ]:
df1['competition_open_since_month'] = df1['competition_open_since_month'].astype(int)
df1['competition_open_since_year'] = df1['competition_open_since_year'].astype(int)

df1['promo2_since_week'] = df1['promo2_since_week'].astype(int)
df1['promo2_since_year'] = df1['promo2_since_year'].astype(int)

## 1.7. Descriptive Statistical

In [ ]:
# separar o dataframe em atributos numéricos e categóricos
num_attributes = df1.select_dtypes(include=['int64', 'float64'])
cat_attributes = df1.select_dtypes(exclude=['int64', 'float64', 'datetime64[ns]'])

## 1.7.1. Numerical Attributes

In [ ]:
# Central Tendency - mean, median
ct1 = pd.DataFrame(num_attributes.apply(np.mean)).T
ct2 = pd.DataFrame(num_attributes.apply(np.median)).T

# Dispersion - std, min, max, range, skew, kurtosis
d1 = pd.DataFrame(num_attributes.apply(np.std)).T
d2 = pd.DataFrame(num_attributes.apply(min)).T
d3 = pd.DataFrame(num_attributes.apply(max)).T
d4 = pd.DataFrame(num_attributes.apply(lambda x: x.max() - x.min())).T
d5 = pd.DataFrame(num_attributes.apply(lambda x: x.skew())).T
d6 = pd.DataFrame(num_attributes.apply(lambda x: x.kurtosis())).T

# concatenate
m = pd.concat([d2, d3, d4, ct1, ct2, d1, d5, d6]).T.reset_index()
m.columns = ['attributes', 'min', 'max', 'range', 'mean', 'median', 'std', 'skew', 'kurtosis']

In [ ]:
m

In [ ]:
sns.displot(df1['sales']);

### 1.7.1. Observações:
<h4>skew</h4>
<ul>
    <li>skew muito positiva: gráfico mais deslocado para a esquerda</li>
    <li>skew muito negativa: gráfico mais deslocado para a direita</li>
</ul>
<h4>kurtosis</h4>
<ul>
    <li>kurtosis alta: maior densidade</li>
    <li>kurtosis baixa: menor densidade</li>
</ul>

## 1.7.2. Categorical Attributes

In [ ]:
# quantidade de valores únicos por variável categórica
cat_attributes.apply(lambda x: x.unique().shape[0])

In [ ]:
aux1 = df1[(df1['state_holiday'] != '0') & (df1['sales'] > 0)]

fig, ax = plt.subplots(1, 3, figsize=(15, 5))

sns.boxplot(x='state_holiday', y='sales', data=aux1, ax=ax[0]);

sns.boxplot(x='store_type', y='sales', data=aux1, ax=ax[1]);

sns.boxplot(x='assortment', y='sales', data=aux1, ax=ax[2]);

# 2.0. Passo 02 - FEATURE ENGINEERING

In [ ]:
df2 = df1.copy()

## 2.1. Mapa Mental de Hipóteses

In [ ]:
Image('img/MindMapHypothesis.png')

## 2.1. Criação das Hipóteses

### 2.1.1. Hipóteses Loja

**1.** Lojas com um quadro maior de funcionários deveriam vender mais.

**2.** Lojas com maior capacidade de estoque deveriam vender mais.

**3.** Lojas com maior porte deveriam vender mais.

**4.** Lojas com maior sortimentos deveriam vender mais.

**5.** Lojas com competidores mais próximos deveriam vender menos.

**6.** Lojas com competidores à mais tempo deveriam vender mais.

### 2.1.2. Hipóteses Produto

**1.** Lojas que investem mais em Marketing deveriam vender mais.

**2.** Lojas com maior exposição de produtos deveriam vender mais.

**3.** Lojas com produtos com menor preço deveriam vender mais.

**4.** Lojas com promoções mais agressivas ( descontos maiores ), deveriam vender mais.

**5.** Lojas com promoções ativas por mais tempo deveriam vender mais.

**6.** Lojas com mais dias de promoção deveriam vender mais.

**7.** Lojas com mais promoções consecutivas deveriam vender mais.

### 2.1.3. Hipóteses Tempo

**1.** Lojas abertas durante o feriado de Natal deveriam vender mais.

**2.** Lojas deveriam vender mais ao longo dos anos.

**3.** Lojas deveriam vender mais no segundo semestre do ano.

**4.** Lojas deveriam vender mais depois do dia 10 de cada mês.

**5.** Lojas deveriam vender menos aos finais de semana.

**6.** Lojas deveriam vender menos durante feriados escolares.

## 2.2. Lista Final de Hipóteses

**1.** Lojas com maior sortimentos deveriam vender mais.

**2.** Lojas com competidores mais próximos deveriam vender menos.

**3.** Lojas com competidores à mais tempo deveriam vender mais.

**4.** Lojas com promoções ativas por mais tempo deveriam vender mais.

**5.** Lojas com mais dias de promoção deveriam vender mais.

**6.** Lojas com mais promoções consecutivas deveriam vender mais.

**7.** Lojas abertas durante o feriado de Natal deveriam vender mais.

**8.** Lojas deveriam vender mais ao longo dos anos.

**9.** Lojas deveriam vender mais no segundo semestre do ano.

**10.** Lojas deveriam vender mais depois do dia 10 de cada mês.

**11.** Lojas deveriam vender menos aos finais de semana.

**12.** Lojas deveriam vender menos durante feriados escolares.

## 2.3. Feature Engineering

In [81]:
# year
df2['year'] = df2['date'].dt.year

# month
df2['month'] = df2['date'].dt.month

# day
df2['day'] = df2['date'].dt.day

# week of year
df2['week_of_year'] = df2['date'].dt.isocalendar().week

# year week
df2['year_week'] = df2['date'].dt.strftime('%Y-%W')

# competition since
df2['competition_since'] = df2.apply(lambda x: datetime.datetime( year=x['competition_open_since_year'], month=x['competition_open_since_month'], day=1), axis=1)
df2['competition_time_month'] = ( (df2['date'] - df2['competition_since']) / 30 ).apply(lambda x: x.days).astype(int)

# promo since
df2['promo_since'] = df2['promo2_since_year'].astype(str) + '-' + df2['promo2_since_week'].astype(str)
df2['promo_since'] = df2['promo_since'].apply(lambda x: datetime.datetime.strptime(x + '-1', '%Y-%W-%w') - datetime.timedelta(days=7))
df2['promo_time_week'] = ( (df2['date'] - df2['promo_since']) / 7).apply(lambda x: x.days).astype(int)

#assortment
df2['assortment'] = df2['assortment'].apply(lambda x: 'basic' if x == 'a' else 'extra' if x == 'b' else 'extended')

# state holiday
df2['state_holiday'] = df2['state_holiday'].apply(lambda x: 'public_holiday' if x == 'a' else 'easter_holiday' if x == 'b' else 'christmas' if x == 'c' else 'regular_day')


In [82]:
df2.head().T

,0,1,2,3,4
store,1,2,3,4,5
day_of_week,5,5,5,5,5
date,2015-07-31 00:00:00,2015-07-31 00:00:00,2015-07-31 00:00:00,2015-07-31 00:00:00,2015-07-31 00:00:00
sales,5263,6064,8314,13995,4822
customers,555,625,821,1498,559
open,1,1,1,1,1
promo,1,1,1,1,1
state_holiday,regular_day,regular_day,regular_day,regular_day,regular_day
school_holiday,1,1,1,1,1
store_type,c,a,a,c,a
